In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("DateFruit_Dataset.csv")

In [3]:
df.head()

,AREA,PERIMETER,MAJOR_AXIS,MINOR_AXIS,ECCENTRICITY,EQDIASQ,SOLIDITY,CONVEX_AREA,EXTENT,ASPECT_RATIO,...,KurtosisRR,KurtosisRG,KurtosisRB,EntropyRR,EntropyRG,EntropyRB,ALLdaub4RR,ALLdaub4RG,ALLdaub4RB,Class
0,422163,2378.908,837.8484,645.6693,0.6373,733.1539,0.9947,424428,0.7831,1.2976,...,3.2370,2.9574,4.2287,-59191263232,-50714214400,-39922372608,58.7255,54.9554,47.8400,BERHI
1,338136,2085.144,723.8198,595.2073,0.5690,656.1464,0.9974,339014,0.7795,1.2161,...,2.6228,2.6350,3.1704,-34233065472,-37462601728,-31477794816,50.0259,52.8168,47.8315,BERHI
2,526843,2647.394,940.7379,715.3638,0.6494,819.0222,0.9962,528876,0.7657,1.3150,...,3.7516,3.8611,4.7192,-93948354560,-74738221056,-60311207936,65.4772,59.2860,51.9378,BERHI
3,416063,2351.210,827.9804,645.2988,0.6266,727.8378,0.9948,418255,0.7759,1.2831,...,5.0401,8.6136,8.2618,-32074307584,-32060925952,-29575010304,43.3900,44.1259,41.1882,BERHI
4,347562,2160.354,763.9877,582.8359,0.6465,665.2291,0.9908,350797,0.7569,1.3108,...,2.7016,2.9761,4.4146,-39980974080,-35980042240,-25593278464,52.7743,50.9080,42.6666,BERHI


In [5]:
df.isnull().sum().sum()

np.int64(0)

In [10]:
x = df.drop("Class",axis = 1)
y = df["Class"]

In [11]:
y.unique()

array(['BERHI', 'DEGLET', 'DOKOL', 'IRAQI', 'ROTANA', 'SAFAVI', 'SOGAY'],
      dtype=object)

In [7]:
from sklearn.model_selection import train_test_split

In [16]:
from sklearn.preprocessing import StandardScaler , LabelEncoder

In [17]:
le = LabelEncoder()
y = le.fit_transform(y)

In [23]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size = 0.2,random_state = 42)

In [20]:
scaler = StandardScaler()

x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

In [21]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader,TensorDataset

In [24]:
x_train_tensor = torch.tensor(x_train_scaled ,dtype = torch.float32)
y_train_tensor = torch.tensor(y_train ,dtype = torch.long)

x_test_tensor = torch.tensor(x_test_scaled ,dtype = torch.float32)
y_test_tensor = torch.tensor(y_test ,dtype = torch.long)

In [25]:
train_dataset = TensorDataset(x_train_tensor , y_train_tensor)
test_dataset = TensorDataset(x_test_tensor , y_test_tensor)

In [26]:
train_loader = DataLoader(train_dataset , batch_size = 32 , shuffle = True)
test_loader = DataLoader(test_dataset , batch_size = 32 )

In [31]:
#  model
class ANN(nn.Module):
    def __init__(self):
        super(ANN,self).__init__()

        self.model = nn.Sequential(
            nn.Linear(x.shape[1],64),
            nn.ReLU(),

            nn.Linear(64,64),
            nn.ReLU(),

            nn.Linear(64,7),
    )

    def forward(self,x):
        return self.model(x)

In [32]:
model = ANN()

criteria = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [35]:
#  training

epochs = 100

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for xb,yb in train_loader:
        optimizer.zero_grad()
        outputs = model(xb)
        loss = criteria(outputs , yb)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
    train_loss = running_loss / len(train_loader)

    print(f"epoch = {epoch}/{epochs}, loss = {train_loss}" )
    

epoch = 0/100, loss = 1.6567400590233181
epoch = 1/100, loss = 1.0492913722991943
epoch = 2/100, loss = 0.6655225572378739
epoch = 3/100, loss = 0.5058224654715994
epoch = 4/100, loss = 0.42665834919266077
epoch = 5/100, loss = 0.38176941871643066
epoch = 6/100, loss = 0.3335141176762788
epoch = 7/100, loss = 0.3021480123633924
epoch = 8/100, loss = 0.2817608416080475
epoch = 9/100, loss = 0.27417566789233166
epoch = 10/100, loss = 0.23339080033094986
epoch = 11/100, loss = 0.22589985669954962
epoch = 12/100, loss = 0.2174609951351
epoch = 13/100, loss = 0.20213947024034418
epoch = 14/100, loss = 0.1987480164869972
epoch = 15/100, loss = 0.1876727954848953
epoch = 16/100, loss = 0.17724507528802622
epoch = 17/100, loss = 0.17316487010406412
epoch = 18/100, loss = 0.17546018958091736
epoch = 19/100, loss = 0.16438979882261026
epoch = 20/100, loss = 0.16009791227786438
epoch = 21/100, loss = 0.14870968679695026
epoch = 22/100, loss = 0.15996137812085773
epoch = 23/100, loss = 0.148962774

In [42]:
#  evalution 
model.eval()
total = 0
correct= 0

with torch.no_grad():
    for xb , yb in test_loader:
        outputs = model(xb)
        _,predicted = torch.max(outputs , 1)

        correct += (predicted == yb).sum().item()
        total += yb.size(0)
print("total_values =  ", total)
print("correct_values =  ", correct)
print("accuracy: ", correct/total*100)

total_values =   180
correct_values =   168
accuracy:  93.33333333333333
